# LangChain Hands-on 02 — Prompt Templates
**Trainer:** Ajeetkumar

**What we will practise**
- Why not plain f-strings?
- PromptTemplate
- ChatPromptTemplate (system + human)
- MessagesPlaceholder (chat history)
- partial variables
- Few-shot templates & example selectors
- Real use cases

> Run on Google Colab. Add your Groq key in Colab **Secrets** (🔑 icon) with the name `groq_key` (free key: console.groq.com).

In [ ]:
! pip install -q langchain langchain-groq

In [ ]:
from google.colab import userdata
groq_api_key = userdata.get('groq_key')

In [ ]:
import os
os.environ['GROQ_API_KEY'] = groq_api_key

In [ ]:
from langchain_groq import ChatGroq

In [ ]:
llm = ChatGroq(model="openai/gpt-oss-20b")

# Disadvantages Without PromptTemplate
- Variables inserted manually → typos, missing values
- Prompts become messy and not reusable
- No roles (system / human)
- Hard to plug into chains

In [ ]:
product = "laptop"
features = "long battery life"
user_query = f"Create a marketing tagline for a {product} that highlights its {features}"
print(llm.invoke(user_query).content)

## 1. PromptTemplate — simple text prompt with variables

In [ ]:
from langchain_core.prompts import PromptTemplate

In [ ]:
template = PromptTemplate(
    input_variables=["adjective", "animal"],
    template="Write a {adjective} 4-line poem about a {animal}"
)

In [ ]:
prompt = template.format(adjective="cute", animal="dog")
prompt

In [ ]:
print(llm.invoke(prompt).content)

In [ ]:
# shortcut: variables are detected automatically
template = PromptTemplate.from_template("Explain {topic} to a {audience} in 3 bullet points")
template.input_variables

In [ ]:
print(llm.invoke(template.format(topic="cloud computing", audience="10 year old kid")).content)

## 2. partial — fix some variables in advance

In [ ]:
template = PromptTemplate.from_template("Translate '{text}' into {language}")
hindi_template = template.partial(language="Hindi")

In [ ]:
hindi_template.format(text="Good morning, how are you?")

In [ ]:
print(llm.invoke(hindi_template.format(text="Good morning, how are you?")).content)

# Benefits of PromptTemplate
- **Reusable** → write once, use with different inputs
- **Easy variable substitution** with `{variable}`
- **Validation** → missing variables raise an error
- **Best for single-shot tasks** (summarise, classify, generate)

## 3. ChatPromptTemplate — prompts with roles

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

In [ ]:
chat_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a kind, helpful assistant."),
    ("human", "Write a {tone} email to a customer about {issue}")
])

In [ ]:
messages = chat_prompt.format_messages(tone="friendly", issue="internet connection issue")
messages

In [ ]:
print(llm.invoke(messages).content)

In [ ]:
# invoke() on a template gives a PromptValue (used in chains later)
prompt_value = chat_prompt.invoke({"tone": "formal", "issue": "late delivery"})
prompt_value.to_messages()

## 4. MessagesPlaceholder — insert chat history into a prompt

In [ ]:
from langchain_core.prompts import MessagesPlaceholder
from langchain.messages import HumanMessage, AIMessage

In [ ]:
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful travel assistant."),
    MessagesPlaceholder("history"),
    ("human", "{question}")
])

In [ ]:
history = [
    HumanMessage("Hi, I am Ajeet. I am planning a trip to Goa."),
    AIMessage("Nice! How many days is your trip?"),
]

In [ ]:
messages = prompt.format_messages(history=history, question="3 days. What is my name and where am I going?")
print(llm.invoke(messages).content)

# Benefits of ChatPromptTemplate
- **Roles** (system / human / ai) → control behaviour
- **Multi-turn** → history with MessagesPlaceholder
- **Works directly in chains and agents**

## 5. Few-shot prompt — teach by examples

In [ ]:
from langchain_core.prompts import FewShotPromptTemplate

In [ ]:
examples = [
    {"word": "happy", "antonym": "sad"},
    {"word": "tall", "antonym": "short"},
    {"word": "fast", "antonym": "slow"},
]
example_prompt = PromptTemplate.from_template("Word: {word}\nAntonym: {antonym}")

In [ ]:
few_shot = FewShotPromptTemplate(
    examples=examples,
    example_prompt=example_prompt,
    prefix="Give the antonym of every word.",
    suffix="Word: {input}\nAntonym:",
    input_variables=["input"],
)
print(few_shot.format(input="rich"))

In [ ]:
print(llm.invoke(few_shot.format(input="rich")).content)

### Few-shot for chat models

In [ ]:
from langchain_core.prompts import FewShotChatMessagePromptTemplate

In [ ]:
examples = [
    {"input": "Rs 250 grocery, usual city", "output": "LOW"},
    {"input": "Rs 95,000 crypto, new device, 3 AM", "output": "HIGH"},
]
example_prompt = ChatPromptTemplate.from_messages([("human", "{input}"), ("ai", "{output}")])
few_shot_chat = FewShotChatMessagePromptTemplate(examples=examples, example_prompt=example_prompt)

In [ ]:
final_prompt = ChatPromptTemplate.from_messages([
    ("system", "Classify transaction fraud risk as LOW, MEDIUM or HIGH. One word."),
    few_shot_chat,
    ("human", "{input}"),
])
print(llm.invoke(final_prompt.format_messages(input="Rs 12,000 electronics, other state, known device")).content)

## 6. Example selector — pick only the examples that fit
`LengthBasedExampleSelector` keeps the prompt short by choosing fewer examples for long inputs.

In [ ]:
from langchain_core.example_selectors import LengthBasedExampleSelector

In [ ]:
examples = [
    {"word": "happy", "antonym": "sad"},
    {"word": "tall", "antonym": "short"},
    {"word": "energetic", "antonym": "lethargic"},
    {"word": "sunny", "antonym": "gloomy"},
]
example_prompt = PromptTemplate.from_template("Word: {word}\nAntonym: {antonym}")
selector = LengthBasedExampleSelector(examples=examples, example_prompt=example_prompt, max_length=25)

In [ ]:
dynamic_prompt = FewShotPromptTemplate(
    example_selector=selector, example_prompt=example_prompt,
    prefix="Give the antonym of every word.", suffix="Word: {input}\nAntonym:", input_variables=["input"])

In [ ]:
print(dynamic_prompt.format(input="big"))

In [ ]:
print(dynamic_prompt.format(input="big and huge and very large and massive and enormous and gigantic"))

# Real use cases
## Customer Support Email

In [ ]:
chat_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a professional customer support assistant."),
    ("human", "Write a {tone} email to {customer_name} about {issue}, offering {resolution}.")
])
messages = chat_prompt.format_messages(tone="apologetic", customer_name="Priya",
                                       issue="late delivery", resolution="a full refund and 10% coupon")
print(llm.invoke(messages).content)

## Interview Question Generator

In [ ]:
template = PromptTemplate.from_template("Generate {n} {level} interview questions on {skill}. Number them.")
print(llm.invoke(template.format(n=5, level="intermediate", skill="Pandas")).content)

## Social Media Caption

In [ ]:
template = PromptTemplate.from_template("Create a {adjective} {platform} caption for {product_name} with 3 hashtags.")
print(llm.invoke(template.format(adjective="fun", platform="Instagram", product_name="EcoBottle")).content)

## 🧪 Practice
1. Make a ChatPromptTemplate "You are a {role}" and test with role = doctor, lawyer, chef.
2. Build a few-shot prompt that converts English sentences to SQL.
3. Use `partial` to fix `company_name` in an HR email template.